# LB-0017 — R2 Pro4-hint SC16 + PAL3 margin1 재현 (A100)

과거 Public LB 0.80144 구성을 재실행하는 전용 노트북입니다. R2 Pro4-hint LoRA로 SC16을 생성하고, SC16 표결 margin이 1 이하인 문항에만 PAL4를 호출하여 실행 결과가 3개 이상 일치할 때만 답을 교체합니다. 4096/8192는 잘린 후보의 감사 기록만 남기며, 최종 답을 바꾸지 않습니다.

In [ ]:
# Cell 1 — Fresh A100 only. Run once, then restart the runtime before Cell 2.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart the runtime once, then run Cells 2–5 in order.")


In [ ]:
# Cell 2 — Mount Drive, obtain the reproducibility repo, and cache the pinned base model.
# GITHUB_TOKEN is needed only while the repository remains private. Store it in Colab Secrets.
from google.colab import drive, userdata
from pathlib import Path
import subprocess

drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = "https://github.com/jhparktime/qwen-math-final-2026.git"
if token:
    url = "https://x-access-token:" + token + "@github.com/jhparktime/qwen-math-final-2026.git"
repo = Path("/content/qwen-math-final")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Freeze the historical R2 SC16 + PAL3-margin1 policy.
import hashlib, json, re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r'[\s_-]+', '', unicodedata.normalize('NFC', str(value)).casefold())

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

roots = [p for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir() and compact(p.name) == compact('2026소중한챌린지')]
assert len(roots) == 1, roots
project = roots[0]
runs = project / 'runs'
LEADERBOARD = project / 'data' / 'deep_chal_math_leaderboard_filtered.csv'
R2_ADAPTER = runs / 'RFT-0004B-r2-pro4-hint-lowdrift-lora' / 'adapter_final'
assert LEADERBOARD.exists(), LEADERBOARD
assert (R2_ADAPTER / 'adapter_config.json').exists()
assert (R2_ADAPTER / 'adapter_model.safetensors').exists()

config = json.loads(Path('configs/final_inference.json').read_text())
config['run_id'] = 'LB-0017-r2pro4-sc16-pal3-margin1-repro'
config['expected_rows'] = 831
config['model']['adapter_name'] = 'r2_pro4hint'
config['model']['adapter_weight_sha256'] = sha256_file(R2_ADAPTER / 'adapter_model.safetensors')
# Exact selection policy: SC16 plurality; PAL only for original SC16 margin <= 1; PAL needs >=3 matching executions.
config['pal']['trigger_margin_le'] = 1
config['pal']['min_agreement'] = 3
# This is a rapid historical-policy reproduction: do not generate longer capped traces.
config['adaptive_length']['audit_capped_rollouts'] = False
config['adaptive_length']['router'].update({'top_gain_min': 99, 'margin_gain_min': 99, 'extended_top_min': 99})
config['adaptive_length']['router']['status'] = 'audit only; final adaptive-length replacement disabled for historical SC16+PAL3 reproduction'
CONFIG_PATH = Path('/content/lb0017_r2_legacy_sc16_pal3_margin1.json')
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
OUTPUT_DIR = runs / config['run_id']
print(json.dumps({
    'leaderboard': str(LEADERBOARD),
    'adapter': str(R2_ADAPTER),
    'adapter_sha256': config['model']['adapter_weight_sha256'],
    'base_revision': config['model']['revision'],
    'sc_samples': config['generation']['n'],
    'pal_trigger': 'original SC16 margin <= 1',
    'pal_agreement': '>= 3 of 4 executed programs',
    'adaptive_final_answer_replacement': False,
    'capped_rollout_audit': False,
    'output_dir': str(OUTPUT_DIR),
}, ensure_ascii=False, indent=2))


In [ ]:
# Cell 4 — Run exactly one R2 inference. Resume-safe. Do not run another vLLM notebook in this runtime.
!PYTHONPATH=. python3 inference/final_inference.py --input {LEADERBOARD} --adapter {R2_ADAPTER} --output-dir {OUTPUT_DIR} --config {CONFIG_PATH}
!python3 scripts/validate_submission.py --input {LEADERBOARD} --submission {OUTPUT_DIR / 'submissions' / 'submission.csv'} --expected-rows 831
EASY_COPY = Path('/content/drive/MyDrive/submission_r2_sc16_pal3_margin1_repro.csv')
EASY_COPY.write_bytes((OUTPUT_DIR / 'submissions' / 'submission.csv').read_bytes())
print('[SUBMIT]', EASY_COPY)


In [ ]:
# Cell 5 — Verify that this was the intended historical policy.
import pandas as pd
report = json.loads((OUTPUT_DIR / 'reports' / 'final_inference_report.json').read_text())
votes = pd.read_csv(OUTPUT_DIR / 'predictions' / 'final_test_vote_diagnostics.csv', dtype=str, keep_default_na=False)
assert report['diagnostics']['adaptive_answer_changes'] == 0, report['diagnostics']
assert votes['used_adaptive_length'].astype(str).str.lower().eq('true').sum() == 0
summary = {
    'run_id': report['run_id'],
    'rows': report['rows'],
    'adapter_sha256': report['adapter_weight_sha256'],
    'sc16_base_vote': 'normalized integer plurality; earliest sample resolves ties',
    'pal_rule': 'original SC16 margin <= 1 and PAL execution agreement >= 3/4',
    'pal_answer_changes': report['diagnostics']['pal_answer_changes'],
    'adaptive_answer_changes': report['diagnostics']['adaptive_answer_changes'],
    'submission': report['artifacts']['submission'],
    'submission_sha256': report['artifacts']['submission_sha256'],
    'easy_copy': str(EASY_COPY),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Final cell — optional GPU release after all artifacts are saved.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True when finished.')
